# Population model ODE — Track C — ICMR-NCDIR totals with a Census-anchored age split

Fits an age-structured population model to the female population of India (or any state) from
**Track C**, and compares 12 ways of representing births, mortality and maturation.

**Track C** = ICMR-NCDIR state totals (2012–2036, extended to 1950–2100) split by Track B's Census-anchored
age distribution. Totals agree with ICMR-NCDIR; the age structure moves with time.

**What to expect.** Close to Track B in age structure, but the total follows ICMR-NCDIR's growth path; where that differs
from the Census/WPP path, cohorts can grow or shrink slightly faster than B's, which the fit has to absorb.

## Model

Sixteen five-year age bands $P_1$ (00–04) … $P_{16}$ (75+):

$$
\begin{aligned}
\frac{dP_1}{dt} &= \Lambda(t) - (k_1 + \mu_1)\,P_1 \\
\frac{dP_i}{dt} &= k_{i-1}P_{i-1} - (k_i + \mu_i)\,P_i, \qquad i = 2,\dots,15 \\
\frac{dP_{16}}{dt} &= k_{15}P_{15} - \mu_{16}\,P_{16}
\end{aligned}
$$

- $\Lambda(t)$: recruitment into 00–04 (births surviving to enter the model), either a **constant** or
  $\Lambda(t) = p\,\Lambda^*(t)$ with a data-driven shape $\Lambda^*(t)$ and one fitted scale $p$.
- $\mu_i$: mortality per band — one **scalar**, a **vector** of 16 fitted rates, or **fixed** at a life table.
- $k_i$: maturation (ageing) rate from band $i$ to $i+1$ — **calibrated** (15 rates) or **fixed at $1/\text{band width} = 1/5$ per year**,
  the standard choice that makes the mean time spent in a band equal to its width.

## Experiments

| | Constant Λ | | | Time-varying Λ(t) | | |
|---|---|---|---|---|---|---|
| | **scalar μ** | **vector μ** | **life-table μ** | **scalar μ** | **vector μ** | **life-table μ** |
| **k calibrated** (Part I) | E1 | E2 | E3 | E4 | E5 | E6 |
| **k = 1/5** (Part II) | E7 | E8 | E9 | E10 | E11 | E12 |

Each Part II experiment is the Part I experiment six numbers lower with $k$ fixed (E7 ↔ E1, …, E12 ↔ E6), so the effect of fixing
$k$ can be read directly from the pairs.

**Fitting.** Weighted least squares over all bands and years: residual = (model − data) / (standard deviation of that band over time).
Bounds: $\Lambda, p \ge 0$; $0 \le \mu_i \le 1$; $0.01 \le k_i \le 1.5$ per year.

**How to read the results.** Each experiment shows all 16 bands (data = small dots every year, model = line, larger dots every 10 years),
then the total, $\Lambda(t)$, the fitted $\mu$ against the life table, and the fitted $k$ against 1/5. Titles give the mean absolute
percentage error (MAPE) across bands, the worst band, and the MAPE of the total. Section 4 compares all 12.

> **Data.** The track files are read from `outputs/track*/` in this repository. They are committed pre-built, so this
> notebook runs without first running notebook 01; re-run notebook 01 to rebuild them.

## 0. Settings and setup

Everything that can be changed is in this cell.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))   # use the repository's src/ when run from notebooks/
import popproj as px
import popmodel as pm
from popproj import sp
warnings.filterwarnings("ignore", category=RuntimeWarning)

TRACK         = "C"
VARIANT       = "B-taper_hold_A1-blend"        # options: `B-taper_hold_A1-blend` (the tag of the saved Track C file)
UNIT          = "India"            # or a state/UT name, e.g. "Kerala"
FIT_START     = 1950               # first year of data used (initial condition)
FIT_END       = 2100
LAMBDA_DRIVER = "w20_29_lead10"    # Λ*(t): "w20_29_lead10" (women 20-29 ten years ahead) or "w15_49"
MAX_NFEV      = 4000               # optimiser budget per fit

TAG = f"track{TRACK}_{VARIANT}_{UNIT.replace(' ', '_')}"
px.configure(fig_dir=os.path.join(px.REPO_ROOT, "figures", "model", TAG))
px.setup_style(scale=0.75)
pd.set_option("display.width", 200, "display.max_columns", 40)
TITLE = f"Track {TRACK} ({VARIANT}), {UNIT}"
print(TITLE, "| figures ->", px.FIG_DIR)

## 1. Data, life table and recruitment driver

- **Population**: the chosen track, years `FIT_START`–`FIT_END`.
- **Life-table mortality**: annual hazards by band from a 22-row life table of death probabilities $q$ (averaged 1950–2100),
  converted as $\mu = -\ln(1-q)/n$ and combined into the 16 bands. Used as fixed $\mu$ (E3, E6, E9, E12) and as the starting
  point for calibrated vectors.
- **Recruitment driver** $\Lambda^*(t)$: built from the same population series, so it carries that track's own timing of births.

In [ ]:
data = pm.load_track(TRACK, VARIANT, UNIT).loc[FIT_START:FIT_END]
mu_lt = pm.life_table_mu(); pm.LT_REF = mu_lt
driver = pm.lambda_driver(pm.load_track(TRACK, VARIANT, UNIT), LAMBDA_DRIVER).loc[data.index]
print(f"{len(data)} years x {data.shape[1]} bands | total {data.index[0]}: {data.iloc[0].sum()/1e6:.1f}M, "
      f"{data.index[-1]}: {data.iloc[-1].sum()/1e6:.1f}M")

fig, axs = plt.subplots(1, 4, figsize=(19, 4.3))
px.shade_icmr_window(axs[0])
axs[0].plot(data.index, data.sum(axis=1), color=sp.L1, lw=2)
px.style_pop_axis(axs[0], "Total female population", ymax=data.sum(axis=1).max())
cmap = plt.get_cmap(sp.CAT)
sh = data.div(data.sum(axis=1), axis=0) * 100
for i, b in enumerate(pm.BANDS):
    axs[1].plot(sh.index, sh[b], color=cmap(i / 15 * 0.9), lw=1.2, label=b)
px.style_pop_axis(axs[1], "Age shares", ylabel="Share [%]"); axs[1].legend(fontsize=6, ncol=2, frameon=False)
axs[2].plot(np.arange(16), mu_lt.values * 1000, color=sp.GREY, marker="s", ms=3)
axs[2].set_yscale("log"); axs[2].set_xticks(range(16)); axs[2].set_xticklabels(pm.BANDS, rotation=60, ha="right", fontsize=7)
axs[2].set_ylabel("μ [per 1000 per yr]"); axs[2].set_title("Life-table mortality"); axs[2].grid(alpha=0.25); sp.lock_ticks(axs[2], "x")
axs[3].plot(driver.index, driver / 1e6, color=sp.OUTC, lw=2)
px.style_pop_axis(axs[3], f"Recruitment driver Λ*(t): {LAMBDA_DRIVER}", ylabel="[millions]")
plt.tight_layout(); px.save(fig, f"{TAG}_data"); plt.show()
pm.experiment_table()

In [ ]:
R = {}   # results of every experiment, filled below

---
## 2. Part I — maturation rates $k_i$ calibrated (E1–E6)

### E1 — Constant Λ, one calibrated μ, calibrated k

The simplest baseline: births and deaths constant, maturation free. Cannot follow the rise and fall of births.

In [ ]:
R["E1"] = pm.fit("E1", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E1"], data, TITLE, fig_name=f"{TAG}_E1")

### E2 — Constant Λ, calibrated μ per band, calibrated k

Age-specific mortality is free; births still constant.

In [ ]:
R["E2"] = pm.fit("E2", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E2"], data, TITLE, fig_name=f"{TAG}_E2")

### E3 — Constant Λ, life-table μ (fixed), calibrated k

Mortality anchored to the life table; only Λ and k are free.

In [ ]:
R["E3"] = pm.fit("E3", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E3"], data, TITLE, fig_name=f"{TAG}_E3")

### E4 — Time-varying Λ(t) = p·Λ*(t), one calibrated μ, calibrated k

Births follow the data-driven driver Λ*(t); one mortality rate for all ages.

In [ ]:
R["E4"] = pm.fit("E4", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E4"], data, TITLE, fig_name=f"{TAG}_E4")

### E5 — Time-varying Λ(t), calibrated μ per band, calibrated k

The most flexible model (32 parameters): an upper bound on fit quality.

In [ ]:
R["E5"] = pm.fit("E5", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E5"], data, TITLE, fig_name=f"{TAG}_E5")

### E6 — Time-varying Λ(t), life-table μ (fixed), calibrated k

Births and mortality anchored to external data; k does all the free work.

In [ ]:
R["E6"] = pm.fit("E6", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E6"], data, TITLE, fig_name=f"{TAG}_E6")

---
## 3. Part II — maturation rates fixed at $k_i = 1/\text{band width} = 1/5$ (E7–E12)

With $k_i = 1/5$ the time spent in each band is exponentially distributed with mean 5 years. This is the usual choice in
age-structured compartmental models; it is only approximate, because an exponential dwell time lets some individuals leave a
band early and others late, which smooths cohort peaks as they move up the ages.

### E7 — Constant Λ, one calibrated μ, k = 1/5

E1 with maturation fixed at 1 / band width.

In [ ]:
R["E7"] = pm.fit("E7", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E7"], data, TITLE, fig_name=f"{TAG}_E7")

### E8 — Constant Λ, calibrated μ per band, k = 1/5

E2 with k fixed: can age-specific mortality compensate for fixed maturation?

In [ ]:
R["E8"] = pm.fit("E8", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E8"], data, TITLE, fig_name=f"{TAG}_E8")

### E9 — Constant Λ, life-table μ, k = 1/5

Only one parameter (Λ): fully physically specified except the level of births.

In [ ]:
R["E9"] = pm.fit("E9", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E9"], data, TITLE, fig_name=f"{TAG}_E9")

### E10 — Time-varying Λ(t), one calibrated μ, k = 1/5

E4 with k fixed.

In [ ]:
R["E10"] = pm.fit("E10", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E10"], data, TITLE, fig_name=f"{TAG}_E10")

### E11 — Time-varying Λ(t), calibrated μ per band, k = 1/5

E5 with k fixed: the most flexible model that keeps k = 1/5.

In [ ]:
R["E11"] = pm.fit("E11", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E11"], data, TITLE, fig_name=f"{TAG}_E11")

### E12 — Time-varying Λ(t), life-table μ, k = 1/5

Only one parameter (p): births shaped by the driver, mortality and maturation fixed.

In [ ]:
R["E12"] = pm.fit("E12", data, mu_lt, driver, max_nfev=MAX_NFEV)
pm.plot_fit(R["E12"], data, TITLE, fig_name=f"{TAG}_E12")

---
## 4. Comparison of all experiments

1. Mean band MAPE and total MAPE for each pair (k calibrated vs k = 1/5), log scale.
2. MAPE for every band × experiment.
3. Calibrated $k_i$ (E1–E6) against 1/5, and calibrated $\mu_i$ against the life table.

In [ ]:
summary = pm.plot_comparison(R, data, TITLE, fig_name=f"{TAG}_comparison")
summary.round(2)

### 4.1 Effect of fixing k: each Part I experiment against its k = 1/5 twin

In [ ]:
pairs = pd.DataFrame({e: {"k calibrated: MAPE %": summary.loc[e, "mean band MAPE %"],
                          "k = 1/5: MAPE %": summary.loc[pm.TWIN[e], "mean band MAPE %"],
                          "k calibrated: total MAPE %": summary.loc[e, "total MAPE %"],
                          "k = 1/5: total MAPE %": summary.loc[pm.TWIN[e], "total MAPE %"],
                          "parameters (k cal / k=1/5)": f"{summary.loc[e, 'params']} / {summary.loc[pm.TWIN[e], 'params']}"}
                      for e in ["E1", "E2", "E3", "E4", "E5", "E6"]}).T
pairs.index = [f"{e} vs {pm.TWIN[e]}: {pm.describe(e).split(': ', 1)[1].rsplit(',', 1)[0]}" for e in pairs.index]
pairs

### 4.2 Save results

Written to `outputs/model/<TAG>/`: summary table, band-by-band MAPE, fitted parameters (JSON) and simulated series (one sheet per experiment).

In [ ]:
print(pm.save_results(R, os.path.join(pm.OUT_DIR, "model", TAG)))

### 4.3 Results (India, C = B-taper split × A1-blend totals, fit 1950–2100)

- Very close to Track B, slightly better: **best band fit** E5 9.6 %; **best total** E4 2.4 % and E5 2.6 %.
- **75+ is the worst band in every experiment** (34–287 % MAPE), for the same reason as Track B: constant mortality cannot follow the
  large growth of the oldest band.
- **Fixing k = 1/5** roughly doubles band error (E5 9.6 % → E11 21.1 %; E4 15.2 % → E10 38.3 %); E12 (19.5 %) again has lower MAPE but
  higher weighted cost than E6 (25.7 %).
- **One-parameter E12** (life-table μ, k = 1/5) gives 19.5 % band error and 8.6 % total error.
- **Parameters at bounds:** calibrated μ vectors put 3–13 of 16 rates at a bound (E2, E5, E8, E11).
- **Likely next improvements:** as for Track B — time-varying mortality, a later fit start, and demographic forcing.